# NAMMA phone render server on a free Colab GPU

Runs the same "Send for render" server as `scripts/serve_render.py` on a Colab T4, so the phone gets a
**photoreal Gaussian splat** (gsplat) instead of the dense points a Mac without a trainer gives.

1. Runtime > Change runtime type > **T4 GPU**.
2. Run the cells in order. The last one prints a link: open it on the phone (or scan the QR code).
3. Record (or open a saved recording), press **Send for render**, and keep this tab open until the result is on the phone.

Free Colab sessions stop after a while; the phone keeps the render id, so "Show my last render" works again while the same session runs.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRANCH = "claude/project-thread-6fvpyc"  # use "main" once the PR is merged
!git clone -q --depth 1 -b $BRANCH https://github.com/RazyAnas/NAMMA.git /content/NAMMA
%cd /content/NAMMA

In [ ]:
# Pipeline dependencies (Colab already has torch + CUDA). gsplat compiles its CUDA kernels the first
# time it trains, which adds a few minutes to the first render only.
!pip -q install "fastapi==0.115.14" "starlette==0.46.2" "uvicorn[standard]" python-multipart pydantic-settings sqlalchemy \
  opencv-python-headless imageio imageio-ffmpeg open3d trimesh "plyfile<1.1.3" "pycolmap==4.2.1" onnxruntime-gpu \
  pyyaml rich httpx qrcode ninja pytorch-msssim "gsplat==1.5.3"
!apt-get -qq install -y ffmpeg > /dev/null
!curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
!python -c "import torch, gsplat, pycolmap, open3d; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| gsplat', gsplat.__version__)"

In [ ]:
# What this machine can do (expect gsplat: true, cuda: true).
!python -c "from app.pipeline.backends import capabilities as c; d=c(); print({k: d[k] for k in ('cuda','gsplat','colmap','onnxruntime','ffmpeg')})"

In [ ]:
# Starts the server + a Cloudflare tunnel and prints the phone link. Leave this cell running;
# stop it with the stop button when you are done.
!python scripts/serve_render.py